# Generate diff mean vector by layer

In [1]:
import json
from pathlib import Path

import pandas as pd
import torch

emb_dir = Path("embeddings_mean_pca_64")
out_path = Path("diffmean_per_layer_mean_pca_64.jsonl")

pt_files = sorted(emb_dir.glob("*.pt"))
print(f"Found {len(pt_files)} embedding files in {emb_dir}")

records = []  # optional: keep in memory as well

with open(out_path, "w", encoding="utf-8") as out_f:
    for path in pt_files:
        print(f"\n=== Processing {path} ===")
        data = torch.load(path, map_location="cpu")

        emb_all = data["embeddings"]              # [N, L, D]
        model_name = data["model_name"]           # full HF name
        benchmark = data["benchmark"]             # e.g. "RQ"
        column = data["column"]                   # e.g. "question_with_context"
        model_short = model_name.split("/")[-1]   # e.g. "Qwen3-4B"

        # load CSV for this benchmark
        csv_path = f"{benchmark}.csv"
        df = pd.read_csv(csv_path)

        print(f"benchmark={benchmark}, column={column}, model={model_name}")
        print(f"embeddings shape: {emb_all.shape}, df rows: {len(df)}")

        # 1) filter to train split
        df_train = df[df["dataset"] == "train"].copy()
        print(f"{benchmark} / {column}: train rows = {len(df_train)}")

        # 2) use `id` to index embeddings
        ids = df_train["id"].to_numpy()
        labels = df_train["binary_label"].to_numpy()  # assumed 0/1

        ids_t = torch.as_tensor(ids, dtype=torch.long)
        assert ids_t.max().item() < emb_all.shape[0], (
            f"Max id {ids_t.max().item()} >= num embeddings {emb_all.shape[0]}"
        )

        X = emb_all[ids_t]                 # [n_train, L, D]
        y = torch.as_tensor(labels, dtype=torch.long)

        n_train, num_layers, embed_dim = X.shape
        print(f"n_train={n_train}, num_layers={num_layers}, embed_dim={embed_dim}")

        # 3) class masks
        pos_mask = (y == 1)
        neg_mask = (y == 0)

        if pos_mask.sum() == 0 or neg_mask.sum() == 0:
            raise ValueError(
                f"{benchmark}/{column}: one of the classes is empty in train split."
            )

        # 4) per-layer means: [L, D]
        mu_pos = X[pos_mask].mean(dim=0)   # [L, D]
        mu_neg = X[neg_mask].mean(dim=0)   # [L, D]
        diffmean = mu_pos - mu_neg         # [L, D]

        print(
            f"{benchmark}/{column}: "
            f"n_pos={pos_mask.sum().item()}, n_neg={neg_mask.sum().item()}"
        )

        record = {
            "model": model_short,
            "model_name": model_name,
            "benchmark": benchmark,
            "column": column,
            "num_layers": int(num_layers),
            "embed_dim": int(embed_dim),
            "diffmean": diffmean.tolist(),  # [L, D] → nested list for JSON
        }

        out_f.write(json.dumps(record) + "\n")
        out_f.flush()

        records.append(record)  # optional
        print(f"Wrote DiffMean for {benchmark}/{column}")

print(f"\nAll done. JSONL saved at {out_path}")


Found 24 embedding files in embeddings_mean_pca_64

=== Processing embeddings_mean_pca_64/RQ_question_Llama-3.1-8B-Instruct.pt ===
benchmark=RQ, column=question, model=meta-llama/Llama-3.1-8B-Instruct
embeddings shape: torch.Size([4997, 33, 64]), df rows: 4997
RQ / question: train rows = 3200
n_train=3200, num_layers=33, embed_dim=64
RQ/question: n_pos=1508, n_neg=1692
Wrote DiffMean for RQ/question

=== Processing embeddings_mean_pca_64/RQ_question_Llama-3.3-70B-Instruct.pt ===
benchmark=RQ, column=question, model=meta-llama/Llama-3.3-70B-Instruct
embeddings shape: torch.Size([4997, 81, 64]), df rows: 4997
RQ / question: train rows = 3200
n_train=3200, num_layers=81, embed_dim=64
RQ/question: n_pos=1508, n_neg=1692
Wrote DiffMean for RQ/question

=== Processing embeddings_mean_pca_64/RQ_question_Qwen3-30B-A3B-Instruct-2507.pt ===
benchmark=RQ, column=question, model=Qwen/Qwen3-30B-A3B-Instruct-2507
embeddings shape: torch.Size([4997, 49, 64]), df rows: 4997
RQ / question: train rows =

# generate the projection

In [2]:
import torch
import pandas as pd
from pathlib import Path

emb_dir = Path("embeddings_mean_pca_64")
proj_dir = Path("projection_mean_pca_64")
proj_dir.mkdir(exist_ok=True)

# Build a lookup: (model_short, benchmark, column) -> diffmean tensor [L, D]
dm_map = {}
for r in records:  # records from the diffmean-per-layer cell
    key = (r["model"], r["benchmark"], r["column"])
    dm_map[key] = torch.tensor(r["diffmean"])  # [L, D]

del records
print(f"Loaded {len(dm_map)} diffmean entries")

pt_files = sorted(emb_dir.glob("*.pt"))
print(f"Found {len(pt_files)} embedding files in {emb_dir}")

for path in pt_files:
    print(f"\n=== Processing {path} ===")
    data = torch.load(path, map_location="cpu")

    emb_all = data["embeddings"]          # [N, L, D]
    model_name = data["model_name"]       # full HF name
    benchmark = data["benchmark"]         # e.g. "RQ"
    column = data["column"]               # e.g. "question_with_context"
    model_short = model_name.split("/")[-1]

    # load the full CSV for this benchmark
    csv_path = f"{benchmark}.csv"
    df = pd.read_csv(csv_path)

    print(f"benchmark={benchmark}, column={column}, model={model_name}")
    print(f"embeddings shape: {emb_all.shape}, df rows: {len(df)}")

    key = (model_short, benchmark, column)
    if key not in dm_map:
        raise KeyError(f"No diffmean found for {key}")

    diffmean = dm_map[key]                # [L, D]

    # Use id to align DF rows with embedding rows
    ids = df["id"].to_numpy()
    labels = df["binary_label"].to_numpy()
    datasets = df["dataset"].tolist()     # keep as list of strings

    ids_t = torch.as_tensor(ids, dtype=torch.long)
    assert ids_t.max().item() < emb_all.shape[0], (
        f"Max id {ids_t.max().item()} >= num embeddings {emb_all.shape[0]}"
    )

    # X: [num_rows, L, D]
    X = emb_all[ids_t]

    num_rows, num_layers, embed_dim = X.shape
    assert diffmean.shape == (num_layers, embed_dim), (
        f"Shape mismatch: X {X.shape}, diffmean {diffmean.shape}"
    )

    print(f"num_rows={num_rows}, num_layers={num_layers}, embed_dim={embed_dim}")

    eps = 1e-8

    # Normalize ONLY the diffmean vectors: d_hat_l = d_l / ||d_l||
    dm_unit = diffmean / (diffmean.norm(dim=-1, keepdim=True) + eps)   # [L, D]

    # Projection: x_{i,l} ⋅ d_hat_l
    # ( [num_rows,L,D] * [1,L,D] ) -> [num_rows,L,D] -> sum over D -> [num_rows,L]
    proj = (X * dm_unit.unsqueeze(0)).sum(dim=-1)   # [num_rows, L]

    print(f"projection shape: {proj.shape}")  # [num_rows, L]

    out = {
        "projection": proj,                          # [num_rows, L]
        "id": torch.as_tensor(ids, dtype=torch.long),
        "binary_label": torch.as_tensor(labels),     # same order as rows in df
        "dataset": datasets,                         # list of split names per row
        "benchmark": benchmark,
        "column": column,
        "model_name": model_name,
    }

    out_path = proj_dir / f"{benchmark}_{column}_{model_short}_projection.pt"
    torch.save(out, out_path)
    print(f"Saved projections to {out_path}")



Loaded 24 diffmean entries
Found 24 embedding files in embeddings_mean_pca_64

=== Processing embeddings_mean_pca_64/RQ_question_Llama-3.1-8B-Instruct.pt ===
benchmark=RQ, column=question, model=meta-llama/Llama-3.1-8B-Instruct
embeddings shape: torch.Size([4997, 33, 64]), df rows: 4997
num_rows=4997, num_layers=33, embed_dim=64
projection shape: torch.Size([4997, 33])
Saved projections to projection_mean_pca_64/RQ_question_Llama-3.1-8B-Instruct_projection.pt

=== Processing embeddings_mean_pca_64/RQ_question_Llama-3.3-70B-Instruct.pt ===
benchmark=RQ, column=question, model=meta-llama/Llama-3.3-70B-Instruct
embeddings shape: torch.Size([4997, 81, 64]), df rows: 4997
num_rows=4997, num_layers=81, embed_dim=64
projection shape: torch.Size([4997, 81])
Saved projections to projection_mean_pca_64/RQ_question_Llama-3.3-70B-Instruct_projection.pt

=== Processing embeddings_mean_pca_64/RQ_question_Qwen3-30B-A3B-Instruct-2507.pt ===
benchmark=RQ, column=question, model=Qwen/Qwen3-30B-A3B-Instr

# calculate AUROC

In [3]:
import json
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import roc_auc_score  # make sure scikit-learn is installed

def auc_confint_hanley_mcneil(y_true, y_score, alpha=0.05):
    """
    y_true: 0/1 labels
    y_score: continuous scores
    Returns: auc, lower, upper  (95% CI by default)
    """
    auc = roc_auc_score(y_true, y_score)

    y_true = np.asarray(y_true)
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()

    Q1 = auc / (2.0 - auc)
    Q2 = 2.0 * auc**2 / (1.0 + auc)

    var_auc = (
        auc * (1.0 - auc)
        + (n_pos - 1.0) * (Q1 - auc**2)
        + (n_neg - 1.0) * (Q2 - auc**2)
    ) / (n_pos * n_neg)

    se = np.sqrt(var_auc)
    z = 1.96  # for 95% CI

    lower = auc - z * se
    upper = auc + z * se

    # clamp to [0,1]
    lower = max(0.0, lower)
    upper = min(1.0, upper)

    return float(auc), float(lower), float(upper)


proj_dir = Path("projection_mean_pca_64")
out_path = Path("auroc_by_layer_mean_pca_64.jsonl")

pt_files = sorted(proj_dir.glob("*.pt"))
print(f"Found {len(pt_files)} projection files in {proj_dir}")

with open(out_path, "w", encoding="utf-8") as out_f:
    for path in pt_files:
        print(f"\n=== Processing {path} ===")
        data = torch.load(path, map_location="cpu")

        proj = data["projection"]            # [N, L]
        labels = data["binary_label"]        # tensor [N]
        datasets = data["dataset"]           # list of length N
        benchmark = data["benchmark"]
        column = data["column"]
        model_name = data["model_name"]
        model_short = model_name.split("/")[-1]

        proj_np = proj.numpy()               # [N, L]
        y_np = labels.numpy()                # [N]

        N, num_layers = proj_np.shape
        print(f"N={N}, num_layers={num_layers}")

        # preserve split order as they first appear in datasets
        seen = set()
        splits = []
        for s in datasets:
            if s not in seen:
                seen.add(s)
                splits.append(s)

        record = {
            "benchmark": benchmark,
            "column": column,
            "model": model_short,
        }

        for split in splits:
            mask = np.array([d == split for d in datasets])
            y_split = y_np[mask]
            X_split = proj_np[mask]          # [n_split, L]

            print(f"  split={split}, n={len(y_split)}")

            aurocs = []
            aurocs_ci = []
            for layer_idx in range(num_layers):
                scores = X_split[:, layer_idx]
                try:
                    auc, lo, hi = auc_confint_hanley_mcneil(y_split, scores)
                except ValueError:
                    # happens if y_split is all 0s or all 1s
                    auc, lo, hi = None, None, None
                aurocs.append(auc)
                aurocs_ci.append([lo, hi])

            # e.g. record["train"] = [auroc_layer0, ...]
            #      record["train_ci"] = [[lo0, hi0], [lo1, hi1], ...]
            record[split] = aurocs
            record[f"{split}_ci"] = aurocs_ci

        out_f.write(json.dumps(record) + "\n")
        out_f.flush()
        print(f"Wrote AUROC record for {benchmark}/{column} ({model_short})")

print(f"\nAll done. AUROCs saved to {out_path}")


Found 24 projection files in projection_mean_pca_64

=== Processing projection_mean_pca_64/RQ_question_Llama-3.1-8B-Instruct_projection.pt ===
N=4997, num_layers=33
  split=train, n=3200
  split=dev, n=797
  split=test, n=1000
Wrote AUROC record for RQ/question (Llama-3.1-8B-Instruct)

=== Processing projection_mean_pca_64/RQ_question_Llama-3.3-70B-Instruct_projection.pt ===
N=4997, num_layers=81
  split=train, n=3200
  split=dev, n=797
  split=test, n=1000
Wrote AUROC record for RQ/question (Llama-3.3-70B-Instruct)

=== Processing projection_mean_pca_64/RQ_question_Qwen3-30B-A3B-Instruct-2507_projection.pt ===
N=4997, num_layers=49
  split=train, n=3200
  split=dev, n=797
  split=test, n=1000
Wrote AUROC record for RQ/question (Qwen3-30B-A3B-Instruct-2507)

=== Processing projection_mean_pca_64/RQ_question_Qwen3-32B_projection.pt ===
N=4997, num_layers=65
  split=train, n=3200
  split=dev, n=797
  split=test, n=1000
Wrote AUROC record for RQ/question (Qwen3-32B)

=== Processing projec